In [19]:
import pandas as pd
import re
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn import metrics
#write your code here


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [20]:
import pandas as pd

import re

import nltk

nltk.download('stopwords')

nltk.download('wordnet')

from nltk.stem import WordNetLemmatizer

from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer

fake = pd.read_csv("https://confrecordings.ams3.digitaloceanspaces.com/Fake1.csv")

true = pd.read_csv("https://confrecordings.ams3.digitaloceanspaces.com/1_True1.csv")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [21]:
print(fake.shape)

print(true.shape)

fake['target'] = 'fake'

true['target'] = 'true'

data = pd.concat([fake, true]).reset_index(drop = True)

from sklearn.utils import shuffle

(49, 4)
(49, 4)


In [22]:
from sklearn.utils import shuffle

data = shuffle(data)
data = data.reset_index(drop = True)
data.drop(["date"], axis=1, inplace=True)
data.drop(["title"], axis=1, inplace=True)
data.drop(["subject"], axis=1, inplace=True)
print(data.head())
wordnet = WordNetLemmatizer()
corpus = []

                                                text target
0  Sunday morning, after what must have seemed li...   fake
1  The following statements were posted to the ve...   true
2  Great Britain s Buckingham Palace announced ye...   fake
3  WASHINGTON (Reuters) - Republican leaders in t...   true
4  WASHINGTON (Reuters) - U.S. House Republicans ...   true


In [23]:
def clean_text(text):
    review = re.sub('[^a-zA-Z]', ' ', text)
    review = review.lower()
    review = review.split()
    review = [wordnet.lemmatize(word) for word in review if word not in stopwords.words('english')]
    review = ' '.join(review)
    return review

In [24]:
data["text"] = data["text"].apply(clean_text)
print(data.head())
x = data["text"]
y = data["target"]

                                                text target
0  sunday morning must seemed like another interm...   fake
1  following statement posted verified twitter ac...   true
2  great britain buckingham palace announced yest...   fake
3  washington reuters republican leader u senate ...   true
4  washington reuters u house republican proposed...   true


In [25]:
print(x)

0     sunday morning must seemed like another interm...
1     following statement posted verified twitter ac...
2     great britain buckingham palace announced yest...
3     washington reuters republican leader u senate ...
4     washington reuters u house republican proposed...
                            ...                        
93    america conversation police brutality black am...
94    breitbart news editor tried use song ringo sta...
95    reuters native american advocate launched elab...
96    washington reuters backing roy moore alabama u...
97    washington reuters trump campaign adviser geor...
Name: text, Length: 98, dtype: object


In [26]:
print(y)

0     fake
1     true
2     fake
3     true
4     true
      ... 
93    fake
94    fake
95    true
96    true
97    true
Name: target, Length: 98, dtype: object


In [27]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

tfidf_v = TfidfVectorizer()
X_tfidf = tfidf_v.fit_transform(X_train)
x_tfidf = tfidf_v.fit_transform(X_test)

In [28]:
X_tfidf.shape

(78, 4442)

In [29]:
x_tfidf.shape

(20, 1749)

In [30]:
# 1. Initialize and Fit the vectorizer on TRAINING data only
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)

# 2. Fit the classifier
classifier1 = LogisticRegression(random_state=0)
classifier1.fit(X_train_tfidf, y_train)

# 3. TRANSFORM (don't fit) the test/new data using the same vectorizer
# Note the lowercase 'x' in your error; ensure this is the transformed version
X_test_tfidf = tfidf.transform(X_test)

# 4. Predict using the correctly shaped features
prediction = classifier1.predict(X_test_tfidf)

print("accuracy: {}%".format(round(accuracy_score(y_test, prediction)*100,2)))

cm = metrics.confusion_matrix(y_test, prediction)

print(cm)

accuracy: 95.0%
[[ 7  1]
 [ 0 12]]


In [31]:
# 1. Train the model using the training features (4,433 features)
classifier2 = tree.DecisionTreeClassifier(random_state=0)
classifier2.fit(X_tfidf, y_train)

# 2. IMPORTANT: Ensure x_tfidf was created using .transform()
# If you did: x_tfidf = vectorizer.fit_transform(X_test) <-- THIS IS THE ERROR
# It should be:
x_tfidf = tfidf.transform(X_test)

# 3. Now the prediction will work because shapes match
prediction = classifier2.predict(x_tfidf)

print("accuracy: {}%".format(round(accuracy_score(y_test, prediction)*100,2)))

cm = metrics.confusion_matrix(y_test, prediction)

print(cm)

accuracy: 100.0%
[[ 8  0]
 [ 0 12]]


In [38]:
classifier3 = RandomForestClassifier(random_state = 0)
classifier3.fit(X_tfidf, y_train)

x_tfidf = tfidf.transform(X_test)

prediction = classifier3.predict(x_tfidf) # Predict on new data

print("accuracy: {}%".format(round(accuracy_score(y_test, prediction)*100,2)))

cm = metrics.confusion_matrix(y_test, prediction)

print(cm)

accuracy: 100.0%
[[ 8  0]
 [ 0 12]]


In [39]:
classifier4 = KNeighborsClassifier(n_neighbors=5)
classifier4.fit(X_tfidf, y_train)

x_tfidf = tfidf.transform(X_test)

prediction = classifier4.predict(x_tfidf) # Predict on new data

print("accuracy: {}%".format(round(accuracy_score(y_test, prediction)*100,2)))

cm = metrics.confusion_matrix(y_test, prediction)

print(cm)

accuracy: 80.0%
[[ 4  4]
 [ 0 12]]


In [42]:
def prediction(news):

    new_xv_test = tfidf.transform([news])

    pred_LR = classifier3.predict(new_xv_test)

    print(pred_LR)

In [43]:
prediction('House Intelligence Committee Chairman Devin Nunes is going to have a bad day. He s been under the assumption, like many of us, that the Christopher Steele-dossier was what prompted the Russia investigation so he s been lashing out at the Department of Justice and the FBI in order to protect Trump. As it happens, the dossier is not what started the investigation, according to documents obtained by the New York Times.Former Trump campaign adviser George Papadopoulos was drunk in a wine bar when he revealed knowledge of Russian opposition research on Hillary Clinton.On top of that, Papadopoulos wasn t just a covfefe boy for Trump, as his administration has alleged. He had a much larger role, but none so damning as being a drunken fool in a wine bar. Coffee boys  don t help to arrange a New York meeting between Trump and President Abdel Fattah el-Sisi of Egypt two months before the election. It was known before that the former aide set up meetings with world leaders for Trump, but team Trump ran with him being merely a coffee boy.In May 2016, Papadopoulos revealed to Australian diplomat Alexander Downer that Russian officials were shopping around possible dirt on then-Democratic presidential nominee Hillary Clinton. Exactly how much Mr. Papadopoulos said that night at the Kensington Wine Rooms with the Australian, Alexander Downer, is unclear,  the report states.  But two months later, when leaked Democratic emails began appearing online, Australian officials passed the information about Mr. Papadopoulos to their American counterparts, according to four current and former American and foreign officials with direct knowledge of the Australians  role. Papadopoulos pleaded guilty to lying to the F.B.I. and is now a cooperating witness with Special Counsel Robert Mueller s team.This isn t a presidency. It s a badly scripted reality TV show.Photo by Win McNamee/Getty Images.')

['fake']


In [44]:
prediction('WASHINGTON (Reuters) - Senate Majority Leader Mitch McConnell said on Monday he was confident the U.S. Congress would be able to reach an agreement to fund the government when the current spending bill ends on Dec. 22 and that there would be no forced government shutdown. â€œThere isnâ€™t any chance we are going to shut the government down. Weâ€™re in discussions, not only on a cap deal, but also on the way forward on appropriations, McConnell told reporters. â€œThe American people need not worry that there is going to be any kind of government shutdown.â€ But U.S. Senate Democratic leader Chuck Schumer said a full-year defense funding bill with short-term money for other programs would fail in the Senate. â€œDemocrats will oppose any budget deal that would allow defense spending to increase while holding down domestic priorities,â€ he said to reporters.  ')

['true']
